## Time-resolved HRV (SDNN / RMSSD)

**Fix:** the previous version computed a *single* SDNN for the whole session and wrote that same number to every row, so the time-resolved HRV plots were flat lines (`nunique(sdnn) == 1`). This computes a **30-beat rolling** SDNN/RMSSD (≈30 s) that varies over the session, keeping the `reltime, datetime, sdnn, rmssd` schema. A 300–2000 ms normal-to-normal filter removes dropped-beat artifacts.

**Inputs:** `data/case-study/processed/ibi*.csv` · **Outputs:** `data/case-study/processed/hrv*.csv`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root -> import mms
import mms
import pandas as pd

PROCESSED = mms.paths.CASE_STUDY / 'processed'
stems = [('ibi', 'hrv')] + [(f'ibi_{s:02d}', f'hrv_{s:02d}') for s in (1, 2, 3)]

for ibi_stem, hrv_stem in stems:
    ibi = pd.read_csv(PROCESSED / f'{ibi_stem}.csv')
    valid = ibi[ibi['ibi'] > 0].copy()
    rolled = mms.hrv.hrv_rolling(valid, window_beats=30)
    out = rolled[['reltime', 'datetime', 'sdnn', 'rmssd']]
    out.to_csv(PROCESSED / f'{hrv_stem}.csv', index=False)
    print(f"{hrv_stem}: {out['sdnn'].nunique()} distinct SDNN values, "
          f"range {out['sdnn'].min():.1f}-{out['sdnn'].max():.1f} ms")
